# Training eines eigenen NER-Modells mit spaCy

Dieses Notebook trainiert ein benutzerdefiniertes Modell für **Named Entity Recognition (NER)** auf englischsprachigen Fahrzeugbewertungen.

Der Ablauf umfasst:

1. Laden und Aufbereiten der Daten  
2. Bereinigung der Texte  
3. Umwandlung manueller Annotationen  
4. Training eines eigenen spaCy-NER-Modells  
5. Visuelle Kontrolle der Ergebnisse  
6. Speicherung des trainierten Modells


## 1. Bibliotheken importieren

Hier werden die benötigten Bibliotheken für Datenverarbeitung, reguläre Ausdrücke, spaCy und das Training des NER-Modells geladen.


In [7]:
# Abschnitt des NER-Trainings
# Benötigte Bibliotheken importieren
import random
from spacy.training import Example
from spacy.util import minibatch
# Benötigte Bibliotheken importieren
import json
# Benötigte Bibliotheken importieren
import re
# Benötigte Bibliotheken importieren
import spacy


## 2. Fahrzeugbewertungen laden und als Textdatei speichern

Die Bewertungen werden aus einer JSON-Datei geladen. Titel und Kommentare werden zusätzlich in eine lesbare Textdatei geschrieben, damit die Daten kontrolliert und weiterverarbeitet werden können.


In [8]:
# Abschnitt des NER-Trainings
title=[]
text=[]
# JSON-Datei mit den gesammelten Bewertungen öffnen
with open("../data/cars_reviews.json", "r", encoding="utf-8") as file:
    data = json.load(file)

# Titel und Kommentare in eine lesbare Textdatei schreiben
with open("../data/cars_reviews.txt", "w", encoding="utf-8") as output_file:
# Über alle Elemente iterieren
    for article in data:
        title = article["Title"]
        text = article["Comment"]

        output_file.write("TITLE:\n")
        output_file.write(title + "\n\n")

        output_file.write("TEXT:\n")
        output_file.write(text + "\n")

        output_file.write("\n" + "-" * 80 + "\n\n")



with open("../data/cars_reviews.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text)


TITLE:
No sound,  smooth riding, comfortable. Best 5 seater electric Vehicle.

TEXT:
The driving experience is awesome. No sound,  smooth riding, comfortable. Best 5 seater. Nice display screen. Feelings awesome after driving this car. The build quality is good. The interior design is perfect.

--------------------------------------------------------------------------------

TITLE:
Dont Buy - After sales is awwful... you will be stuck

TEXT:
Me and my colleague bought BYD Atto 3 months ago. Its a wondeful car to drive and we were very happy unless the car met with an accident.It require change in back light and some work on truck. No internal damage was done. It has been over 4 weeks, BYD is unable to repair the car.They only have 1 reason, part is not available.I would recommend not to buy as you will be stuck for after sales support and parts.

--------------------------------------------------------------------------------

TITLE:
The BYD Atto 3 claims to offer a distance of 420km



## 3. Vortrainiertes spaCy-Modell laden

Das englische Modell `en_core_web_sm` wird geladen. Es dient als Vergleich, um zu prüfen, welche Entitäten spaCy bereits ohne eigenes Training erkennt.


In [9]:
# Abschnitt des NER-Trainings
# Vortrainiertes englisches spaCy-Modell laden
nlp = spacy.load("en_core_web_sm")


## 4. Entitäten des vortrainierten Modells anzeigen

Eine Beispielbewertung wird analysiert und die erkannten Entitäten werden mit `displacy` farbig dargestellt.


In [10]:

# Abschnitt des NER-Trainings
doc = nlp(data[2]["Comment"])
spacy.displacy.render(doc, style='ent')


## 5. Inline-Annotationen in das spaCy-Format umwandeln

Die Trainingsdaten enthalten Markierungen wie `[engine]ENGINE`. Die Funktion entfernt diese Markierungen aus dem Text und erzeugt daraus Entitätspositionen im spaCy-Format.


In [11]:

# Abschnitt des NER-Trainings
def annotation(txt):
    """Konvertiert Inline-Annotationen in das spaCy-Trainingsformat."""
    ent =  []
    while 1:
        pos1=txt.find("[")
        if pos1 < 0:
            break;
        pos2=txt.find("]",pos1)
        pos3=pos2+2
        while pos3 < len(txt) and txt[pos3:pos3+1].isalpha():
            pos3 += 1

        e = txt[pos1+1:pos2]
        t = txt[pos2+2:pos3]        
        txt = txt[:pos1] + e + txt[pos3:]
        if (t == "ohne"):
            print(e, t)
        
        ent.append((pos1, pos2-1, t))
        #txt = txt[0:pos1] +
        #print(txt)
    return txt, {"entities": ent}


## 6. Texte bereinigen

URLs, E-Mail-Adressen, Telefonnummern, längere Zahlenfolgen und überflüssige Leerzeichen werden entfernt. Dadurch wird der Text für das Training vereinheitlicht.


In [12]:
# Abschnitt des NER-Trainings
def clean_text(text):
    """Bereinigt einen Text vor dem Modelltraining."""
    
    # URLs und Webadressen entfernen
    text = re.sub( r'https?://\S+|www\.\S+','',text)

    # E-Mail-Adressen entfernen
    text = re.sub(r'\b[\w.-]+@[\w.-]+\.\w+\b','',text)

    # Telefonnummern, Datumsangaben und längere Zahlenfolgen entfernen
    text = re.sub(r'(?:\+?\d[\d\s()/.-]{6,}\d)','',text)

    # Mehrfache Leerzeichen vereinheitlichen
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


## 7. Leeres NER-Modell erstellen

Eine leere englische spaCy-Pipeline wird erzeugt und um die Komponente `ner` erweitert.


In [13]:
# Abschnitt des NER-Trainings
nlp_c = spacy.blank("en")
ner_c= nlp_c.add_pipe("ner")


## 8. Manuell annotierte Trainingsdaten laden

Die Datei mit den Inline-Annotationen wird geladen. Die einzelnen Bewertungen werden anhand langer Trennlinien voneinander getrennt.


In [14]:
# Abschnitt des NER-Trainings
# Datei mit den manuell annotierten Texten laden
with open("../data/cars_reviews_ner_inline.txt", "r", encoding="utf-8") as file:
    text_ner = file.read()


# Einzelne Bewertungen anhand langer Trennlinien aufteilen
articles = re.split(r"\n-{10,}\n", text_ner)
articles = [a.strip() for a in articles if a.strip()]

print(len(articles))


174


## 9. Annotierte Texte bereinigen

Jeder annotierte Text wird mit der zuvor definierten Bereinigungsfunktion verarbeitet.


In [15]:
# Abschnitt des NER-Trainings
articles_clean = []

# Über alle Elemente iterieren
for article in articles:
    clean_article = clean_text(article)
    articles_clean.append(clean_article)


## 10. Annotationen konvertieren

Alle bereinigten Texte werden in das von spaCy erwartete Trainingsformat umgewandelt.


In [16]:
# Abschnitt des NER-Trainings
annotations = [annotation(t) for t in articles_clean]
annotations


[('TITLE:\nNo sound, smooth riding, comfortable. Best 5 seater electric Vehicle.\n\nTEXT:\nThe driving experience is awesome. No sound, smooth riding, comfortable. Best 5 seater. Nice display screen. Feelings awesome after driving this car. The build quality_QUALITY is good. The interior design is perfect.',
  {'entities': [(88, 106, 'PERFORMANCE'),
    (144, 155, 'COMFORT'),
    (177, 184, 'FEATURE'),
    (238, 251, 'BUILD'),
    (273, 281, 'INTERIOR'),
    (282, 288, 'DESIGN')]}),
 ('TITLE:\nDont Buy - After sales is awwful... you will be stuck\n\nTEXT:\nMe and my colleague bought BYD Atto 3 months ago. Its a wondeful car to drive and we were very happy unless the car met with an accident.It require change in back light and some work on truck. No internal damage was done. It has been over 4 weeks, BYD is unable to repair the car.They only have 1 reason, part is not available_PART.I would recommend not to buy as you will be stuck for after sales support and parts_PART.',
  {'entities'

## 13. Entitätspositionen kontrollieren

Die markierten Textausschnitte und ihre Labels werden ausgegeben. So können falsche Positionen oder fehlerhafte Annotationen erkannt werden.


In [16]:
# Abschnitt des NER-Trainings
# Über alle Elemente iterieren
for i in range(len(articles_clean)):
# Über alle Elemente iterieren
    for e in annotations[i][1]["entities"]:
        print("'" + annotations[i][0][e[0]:e[1]]+ "'",e[2])


'driving experience' PERFORMANCE
'comfortable' COMFORT
'display' FEATURE
'build quality' BUILD
'interior' INTERIOR
'design' DESIGN
'damage' REPAIR
'repair' REPAIR
'part is not available' SPARE
'after sales support' SERVICE
'parts' SPARE
'420km' RANGE
'300km' RANGE
'charge' CHARGING
'27km' RANGE
'54km' RANGE
'battery charge' BATTERY
'battery' BATTERY
'battery' BATTERY
'battery' BATTERY
'charge' CHARGING
'battery charge' BATTERY
'kWh' BATTERY
'Worst' PROBLEM
'performance' PERFORMANCE
'battery' BATTERY
'range' RANGE
'Look' DESIGN
'color' DESIGN
'dent' REPAIR
'service' SERVICE
'WORST' PROBLEM
'PART' SPARE
'color' DESIGN
'color' DESIGN
'color' DESIGN
'dent' REPAIR
'1.2LAKHS' PRICE
'replaced' REPAIR
'fix' REPAIR
'service' SERVICE
'showroom' SERVICE
'parts were not available' SPARE
'showroom' SERVICE
'parts' SPARE
'parts' SPARE
'parts' SPARE
'customer care' SERVICE
'space' SPACE
'showroom' SERVICE
'repair' REPAIR
'part' SPARE
'part' SPARE
'parts' SPARE
'comfort' COMFORT
'safety' SAFETY
'perfo

## 14. Verwendete Entitätsklassen ermitteln

Alle unterschiedlichen Labels werden gesammelt und ausgegeben. Unerwartete oder falsch geschriebene Klassen lassen sich dadurch leichter erkennen.


In [17]:
# Abschnitt des NER-Trainings
entities = set()
# Über alle Elemente iterieren
for a in annotations:
    #print(a[1])
# Über alle Elemente iterieren
    for ent in a[1].get('entities'):
        entities.add(ent[2])
print("Gefunden Entitäten:", entities)


Gefunden Entitäten: {'PRICE', 'HANDLING', 'RANGE', 'SERVICE', 'BATTERY', 'VALUE', 'ENGINE', 'COMFORT', 'ADAS', 'SPARE', 'PROBLEM', 'SPACE', 'FUEL', 'CHARGING', 'DESIGN', 'INTERIOR', 'AIRBAG', 'PERFORMANCE', 'FEATURE', 'SEAT', 'BUILD', 'REPAIR', 'SAFETY'}


## 15. spaCy-Trainingsbeispiele erzeugen

Aus jedem Text und seinen Annotationen wird ein `Example`-Objekt erstellt. Anschließend wird das leere Modell mit den vorhandenen Labels initialisiert.


In [18]:
# Abschnitt des NER-Trainings
examples = []
# Über alle Elemente iterieren
for text, annots in annotations:
    examples.append(Example.from_dict(nlp.make_doc(text), annots))
#print(examples)
nlp_c.initialize(lambda: examples) # Initialisierung ist nur bei einem leeren Modell erforderlich

print("Anzahl Examples:", len(examples))
# Über alle Elemente iterieren
print("Längster Text:", max(len(ex.reference.text) for ex in examples))


c:\master_ai\nlp\projeckt\NLP_Projekt\.venv\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "TITLE:
No sound, smooth riding, comfortable. Best ..." with entities "[(88, 106, 'PERFORMANCE'), (144, 155, 'COMFORT'), ...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\master_ai\nlp\projeckt\NLP_Projekt\.venv\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "TITLE:
Dont Buy - After sales is awwful... you wil..." with entities "[(273, 279, 'REPAIR'), (333, 339, 'REPAIR'), (373,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\master_ai\nlp\projeckt\NLP_Projekt\.venv\Lib\site-packages\spacy\training

Anzahl Examples: 174
Längster Text: 2493


## 16. Eigenes NER-Modell trainieren

Das Modell wird über 50 Epochen trainiert. Die Beispiele werden vor jeder Epoche gemischt und in Mini-Batches verarbeitet. Der Loss wird nach jeder Epoche ausgegeben.


In [26]:

# Abschnitt des NER-Trainings
# Über alle Elemente iterieren
# Das Modell über 50 Epochen trainieren
for i in range(50):
    # Trainingsbeispiele vor jeder Epoche zufällig mischen
    random.shuffle(examples)
    losses = {}
# Über alle Elemente iterieren
    # Trainingsdaten in Mini-Batches verarbeiten
    for batch in minibatch(examples, size=8):
        nlp_c.update(batch, losses=losses)
    print(i, losses)


0 {'ner': np.float32(8063.7793)}
1 {'ner': np.float32(2012.6846)}
2 {'ner': np.float32(1390.2637)}
3 {'ner': np.float32(1071.3234)}
4 {'ner': np.float32(831.38226)}
5 {'ner': np.float32(686.5195)}
6 {'ner': np.float32(346.77924)}
7 {'ner': np.float32(226.109)}
8 {'ner': np.float32(141.6993)}
9 {'ner': np.float32(97.68001)}
10 {'ner': np.float32(152.18776)}
11 {'ner': np.float32(68.454926)}
12 {'ner': np.float32(60.84521)}
13 {'ner': np.float32(66.178154)}
14 {'ner': np.float32(29.577684)}
15 {'ner': np.float32(35.45264)}
16 {'ner': np.float32(34.611305)}
17 {'ner': np.float32(27.86856)}
18 {'ner': np.float32(26.250462)}
19 {'ner': np.float32(22.621695)}
20 {'ner': np.float32(23.270998)}
21 {'ner': np.float32(24.570189)}
22 {'ner': np.float32(22.830465)}
23 {'ner': np.float32(10.778326)}
24 {'ner': np.float32(9.698778)}
25 {'ner': np.float32(29.234852)}
26 {'ner': np.float32(14.367422)}
27 {'ner': np.float32(22.752514)}
28 {'ner': np.float32(23.002848)}
29 {'ner': np.float32(14.960401)}

## 17. Trainiertes Modell testen

Eine Beispielbewertung wird mit dem selbst trainierten Modell analysiert und die erkannten Entitäten werden visualisiert.


In [27]:
# Abschnitt des NER-Trainings
doc = nlp_c("Good looking seats in more comfortable overall features all best compare to this price range. This is the first different car range, also good compared to all EV car this price point of view, no one can beat the MG Windsor in terms of design and features.")
spacy.displacy.render(doc, style='ent')


## 18. Modell speichern

Das fertige NER-Modell wird im Projektordner gespeichert und kann später in anderen Notebooks wieder geladen werden.


In [ ]:
# Abschnitt des NER-Trainings
# Trainiertes Modell dauerhaft speichern
nlp_c.to_disk("../models/my_ner_model")
